# Tutorial to use ImputeGAP

---
## 1. Introduction

### 1.1. Creating and visualizing an existent dataset

#### Create a timeseries object, load a series, normalize it

If I correctly understood the .txt series is a 2d representation of the dataset, each column is a channel, each row is a timestep, and then if you have multiple samples, you simply concatenate them as new rows. Then it is up to me to choose nbr_val to cut the samples.

In this case eeg-alcohol is a an individual sample, with 64 channels (features, time series) and 256 values (time length).

In [1]:
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset from the library
ts.load_series(utils.search_path("eeg-alcohol"))
ts.normalize(normalizer="z_score")

# print and plot a subset of time series
ts.print(nbr_series=6, nbr_val=20)
ts.plot(input_data=ts.data, nbr_series=6, nbr_val=100, save_path="./imputegap_assets")


(SYS) The dataset is loaded from /mnt/fast/projects/ds_sem_repos/imputegap_repo/ImputeGAP/imputegap/datasets/eeg-alcohol.txt

> logs: normalization (z_score) of the data - runtime: 0.0008 seconds

shape of eeg-alcohol : (64, 256)
	number of series = 64
	number of values = 256

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           
idx_0             2.1903285853   1.6705262632   2.2587543576   2.2311576144   2.0642608690   2.0352202783
idx_1             1.9472603285   1.5655748037   2.1966398110   2.2311576144   1.9054970599   1.9505308899
idx_2             1.7041920717   1.4605158119   2.1966398110   2.3667495605   1.7468957521   1.7814984911
idx_3             1.5825957138   1.3029810904   2.1345252645   2.3667495605   1.5088312891   1.6122929032
idx_4             1.4003878654   1.0929706391   1.8858125103   2.1633616414   1.3500674799   1.3585711159
idx_5             1.2180555580   0.7779011960   1.5751124936   1.7563079509   1.19

'./imputegap_assets/25_12_16_16_12_08_imputegap_plot.jpg'

In [2]:
# (T, V), T: time series, V: values
ts.data.shape

(64, 256)

#### List of all the datasets available

In [3]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"ImputeGAP datasets : {ts.datasets}")

ImputeGAP datasets : ['airq', 'bafu', 'chlorine', 'climate', 'drift', 'eeg-alcohol', 'eeg-reading', 'electricity', 'fmri-stoptask', 'forecast-economy', 'meteo', 'motion', 'soccer', 'solar-plant', 'sport-activity', 'stock-exchange', 'temperature', 'traffic']


### 1.2. Contamination

In [4]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Missingness patterns : {ts.patterns}")

Missingness patterns : ['aligned', 'disjoint', 'distribution', 'gaussian', 'mcar', 'overlap', 'scattered']


### 1.3. Imputation

I had to install this on my ubuntu machine:
```bash
sudo apt-get install libopenblas0 libopenblas-dev
```

In [5]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Imputation families : {ts.families}")
print(f"Imputation algorithms : {ts.algorithms}")

Imputation families : ['DeepLearning', 'LLMs', 'MachineLearning', 'MatrixCompletion', 'PatternSearch', 'Statistics']
Imputation algorithms : ['BRITS', 'BayOTIDE', 'BitGraph', 'CDRec', 'DeepMVI', 'DynaMMo', 'GAIN', 'GPT4TS', 'GRIN', 'GROUSE', 'HKMF_T', 'IIM', 'Interpolation', 'IterativeSVD', 'KNNImpute', 'MICE', 'MPIN', 'MRNN', 'MeanImpute', 'MeanImputeBySeries', 'MinImpute', 'MissForest', 'MissNet', 'NuwaTS', 'PRISTI', 'ROSL', 'SPIRIT', 'STMVL', 'SVT', 'SoftImpute', 'TKCM', 'TRMF', 'XGBOOST', 'ZeroImpute']


### 1.4. Others (still to check)

#### Parameter Tuning

In [6]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"AutoML Optimizers : {ts.optimizers}")

AutoML Optimizers : ['bayesian', 'greedy', 'particle_swarm', 'ray_tune', 'successive_halving']


#### Benchmark

In [7]:
from imputegap.recovery.benchmark import Benchmark

my_algorithms = ["SoftImpute", "MeanImpute"]

my_opt = ["default_params"]

my_datasets = ["eeg-alcohol"]

my_patterns = ["mcar"]

range = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8]

my_metrics = ["*"]

# launch the evaluation
bench = Benchmark()
bench.eval(algorithms=my_algorithms, datasets=my_datasets, patterns=my_patterns, x_axis=range, metrics=my_metrics, optimizers=my_opt)


(SYS) The dataset is loaded from /mnt/fast/projects/ds_sem_repos/imputegap_repo/ImputeGAP/imputegap/datasets/eeg-alcohol.txt

SoftImpute is tested with mcar, started at 2025-12-16 16:12:34.
done!


MeanImpute is tested with mcar, started at 2025-12-16 16:12:43.
done!



> logs: benchmark - Execution Time: 17.0835 seconds


eegalcohol: {mcar, RMSE, default_params}

 Rate       MeanImpute          SoftImpute     

 0.05      1.1073947986        0.4359915238    
  0.1      0.8569349077        0.3665001858    
  0.2      0.9924113085        0.3983300622    
  0.4      1.0058063455        0.4355910162    
  0.6      0.9891809506        0.4500113662    
  0.8      0.9927953863        0.4655442240    



eegalcohol: {mcar, RUNTIME[ms], default_params}

 Rate       MeanImpute          SoftImpute     

 0.05      5.8341026306       37.7695560455    
  0.1      1.0000000000       36.5927219391    
  0.2      1.0000000000       10.5044841766    
  0.4      1.0000000000       10.9333992004    
  

---
## 2. GPVAE Integration

For the integration, the datasets have been merge in individual .txt files for Physionet, and individual .npy files for both HMNIST and SPRITES. Train, val and test splits
have been merged in a individual file, and then in a config.yaml it is possible to specify the splits' lengths.

In npy files we have stored them as (T, V) np.arrays, T being the time series (channels) and V being the values, in this case having multiple independent samples, (T, S*V).

Example of how full train and test (no missingess) have been concatenated:

```python
import numpy as np
test = np.load('external/hmnist/hmnist_mnar.npz')
x_train_full = test['x_train_full']
x_test_full = test['x_test_full']

x_full = np.concatenate([x_train_full, x_test_full], axis=0)
np.save('external/hmnist/test.npy', x_full.transpose(2,0,1).reshape(784, -1))
```

Example of how miss_train and miss_test (with missing values, authors' contamination) have been concatenated:

```python
x_train_miss = test['x_train_miss']
m_train_miss = test['m_train_miss']
x_test_miss = test['x_test_miss']
m_test_miss = test['m_test_miss']

# 1s in the missing mask represent position of missing values (and they are 0.0)
np.unique(x_train_miss[m_train_miss==1])

# set missing values to NaN, indeed in the integration the gpvae expects to get incomp data with NaN, and then it 
# creates the mask in function of that and finally replace NaN with 0.0
x_train_miss_nan = x_train_miss.copy()
x_train_miss_nan[m_train_miss==1] = np.nan
x_test_miss_nan = x_test_miss.copy()
x_test_miss_nan[m_test_miss==1] = np.nan

# concatenate train and test (S_train + S_test, V, T)
x_miss_nan = np.concatenate(x_train_miss_nan, x_test_miss_nan)

# save the concatenate train and test miss (with nans)
np.save('external/hmnist/test.npy', x_miss_nan.transpose(2,0,1).reshape(784, -1))
```

---
Utils

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

def plot_training_results(path):
  with open(os.path.join(path, 'training_curve.tsv'), 'r') as infile:
      df = pd.read_csv(infile, sep='\t')

  # Filter to fetch only rows where also validation losses were computed
  no_nan = df.loc[np.invert(df['val_loss'].isna())]

  _, axes = plt.subplots(1, 3, figsize=(10, 4), layout='constrained')
  axes[0].plot(df.index, df['train_loss'], label='Train')
  axes[0].plot(no_nan.index, no_nan['val_loss'], label='Validation')
  axes[0].legend(loc='upper right')
  axes[0].set_title("Train and validation loss")
  axes[0].set_ylabel("Loss: NLL + beta*KL")
  axes[0].set_xlabel("Training steps")
  axes[1].plot(df.index, df['train_nll'], label='Train')
  axes[1].plot(no_nan.index, no_nan['val_nll'], label='Validation')
  axes[1].legend(loc='upper right')
  axes[1].set_title("Train and validation NLL")
  axes[1].set_ylabel("NLL")
  axes[1].set_xlabel("Training steps")
  axes[2].plot(df.index, df['train_kl'], label="Train")
  axes[2].plot(no_nan.index, no_nan['val_kl'], label="Validation")
  axes[2].legend(loc='upper right')
  axes[2].set_title("Train and validation KL")
  axes[2].set_ylabel("KL")
  axes[2].set_xlabel("Training steps")
  plt.show()


# for computer vision datasets (HMNIST and SPRITES) imputation visualization
def create_imputation_plot(image_shape, time_length, miss, imputed_no_gt, imputed, gt, sample_id, figsize=(15,5), cmap="gray"):
  miss[np.isnan(miss)] = 0.0
  fig, axes = plt.subplots(4, time_length, figsize=figsize, layout='constrained', sharey=True)

  for j in range(time_length):
    axes[0, j].imshow(np.clip(miss[sample_id,j].reshape(*image_shape), 0, 1), cmap=cmap)
    axes[1, j].imshow(np.clip(imputed_no_gt[sample_id,j].reshape(*image_shape), 0, 1), cmap=cmap)
    axes[2, j].imshow(np.clip(imputed[sample_id,j].reshape(*image_shape), 0, 1), cmap=cmap)
    axes[3, j].imshow(np.clip(gt[sample_id,j].reshape(*image_shape), 0, 1), cmap=cmap)
    axes[0,j].axis('off')
    axes[1,j].axis('off')
    axes[2,j].axis('off')
    axes[3,j].axis('off')
  axes[0,0].set_ylabel("Miss")
  fig.suptitle("Miss -> Imputed (no GT) -> Imputed | GT")
  plt.show()

### 2.1 Train the GPVAE model with missingess patterns provided by the author

#### 2.1.1. Physionet

In [9]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/physionet/merged/x_miss_nan.txt")
# ts.normalize(normalizer="z_score")

ts_m_imputed = gp_vae(ts.data, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_physionet.yaml", 
                      batch_size=64, epoch=2, verbose=False)


(SYS) The dataset is loaded from ./external/datasets/physionet/merged/x_miss_nan.txt



: 

In [ ]:
# plot training results
exp_folder = 'imputegap_assets/models/20251205_151509'
plot_training_results(exp_folder)

Reload model and impute one sample

In [1]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# 28th sample (index 27)
sample_idx = 27
seq_length = 48
nbr_features = 35
flatten_idx = sample_idx*seq_length

# model folder
# model_folder = 'imputegap_assets/models/20251205_151509'
model_folder = 'external/models/251123_reproduce_physionet'

# import single sample
ts.load_series('./external/datasets/physionet/merged/x_full.txt')
# load and normalize the dataset
# ts.normalize(normalizer="z_score")

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+seq_length], 
                             rate_dataset=0.4, 
                             rate_series=0.2, 
                             block_size=7, 
                             seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+seq_length], ts_m, subplot=True)

ts_m_imputed = gp_vae(ts_m, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_physionet.yaml", 
                      model_folder, 
                      verbose=True)

ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+seq_length], incomp_data=ts_m, recov_data=ts_m_imputed, nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")

2025-12-16 16:16:11.000831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-16 16:16:11.036857: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-16 16:16:11.036895: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-16 16:16:11.058840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-16 16:16:13.346470: W tensorflow/compiler/tf


(SYS) The dataset is loaded from ./external/datasets/physionet/merged/x_full.txt


(CONT) missigness pattern: MCAR
	selected series: 1, 5, 9, 10, 13, 14, 16, 17, 20, 22, 25, 27, 30, 34
	percentage of contaminated series: 40.0%
	rate of missing data per series: 20.0%
	block size: 7
	security offset: [0-4]
	seed value: 42

plots saved in: ./imputegap_assets/25_12_16_16_16_35_imputegap_plot.jpg


2025-12-16 16:16:42.277180: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-16 16:16:42.330719: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (1, 48, 128)           │       107,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 48, 128)           │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, 48, 105)           │        13,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Decoder:  None
Checkpoint successfully restored.
Starting imputation...


Imputing progress: 100%|██████████| 1/1 [00:00<00:00, 21.37it/s]


> logs: Imputing with gpvae - Execution Time: 0.0574 seconds


> logs: imputation gpvae - Execution Time: 0.4925 seconds




plots saved in: ./imputegap_assets/imputation/25_12_16_16_16_52_GP-VAE_plot.jpg


'./imputegap_assets/imputation/25_12_16_16_16_52_GP-VAE_plot.jpg'

---
#### 2.1.2. HMNIST

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
hmnist_nan = np.load("./external/datasets/hmnist/x_miss_nan.npy")
ts.import_matrix(hmnist_nan)
# ts.normalize(normalizer="z_score")

ts_m_imputed = gp_vae(ts.data, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      batch_size=64, epoch=2, verbose=True)

In [ ]:
# plot training results
exp_folder = 'imputegap_assets/models/20251205_152733'
plot_training_results(exp_folder)

Reload model and impute one sample

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

# 28th sample (index 27)
sample_idx = 27
seq_length = 10
nbr_features = 784
flatten_idx = sample_idx*seq_length

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'

hmnist_full = np.load("./external/datasets/hmnist/x_full.npy")

# load and normalize the dataset
ts.import_matrix(hmnist_full)
# ts.normalize(normalizer="z_score")

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+seq_length], 
                             rate_dataset=1.0, 
                             rate_series=0.8, 
                             block_size=7, 
                             seed=True)

ts_m_imputed = gp_vae(ts_m, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder, 
                      verbose=True)

In [ ]:
create_imputation_plot((28,28,1), 
                       10, 
                       np.expand_dims(ts_m.T, axis=0),
                       np.expand_dims(ts_m_imputed.T, axis=0),
                       np.expand_dims(ts_m_imputed.T, axis=0),
                       np.expand_dims(ts.data[:,flatten_idx:flatten_idx+seq_length].T, axis=0),
                       0, cmap="gray")

---
#### 2.1.3. Sprites

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
# to avoid memory overflow I slice the dataset and change splits in the config
# accordingly
sprites_nan = np.load("./external/datasets/sprites/x_miss_nan.npy")[:,:10_000]
ts.import_matrix(sprites_nan)
# ts.normalize(normalizer="z_score")

ts_m_imputed = gp_vae(ts.data, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_sprites.yaml", 
                      batch_size=64, epoch=2, verbose=True)

In [ ]:
# plot training results
exp_folder = 'imputegap_assets/models/20251205_204940'
plot_training_results(exp_folder)

Reload a model and impute

With sprites the issue is that the reconstruction works well if the contamination is done in a way that we either remove all three channels for a pixel, or keep them all.

In [ ]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

# 28th sample (index 27)
sample_idx = 27
seq_length = 8
nbr_features = 12288
flatten_idx = sample_idx*seq_length

# model folder
# model_folder = 'imputegap_assets/models/20251205_204940'
model_folder = 'external/models/251114_reproduce_sprites'

sprites_full = np.load("./external/datasets/sprites/x_full.npy")[:,:10000]

# load and normalize the dataset
ts.import_matrix(sprites_full)
# ts.normalize(normalizer="z_score")

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+seq_length], 
                             rate_dataset=1.0, 
                             rate_series=0.8, 
                             block_size=3, 
                             seed=True)

ts_m_imputed = gp_vae(ts_m, 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_sprites.yaml", 
                      model_folder, 
                      verbose=True)

In [ ]:
create_imputation_plot((64,64,3),
                       8, 
                       np.expand_dims(ts_m.T, axis=0),
                       np.expand_dims(ts_m_imputed.T, axis=0),
                       np.expand_dims(ts_m_imputed.T, axis=0),
                       np.expand_dims(ts.data[:,flatten_idx:flatten_idx+seq_length].T, axis=0),
                       0, cmap=None)

## 3. New Experiment

Keeping the same trained model on MNAR (reproducibility experiment). MNAR: Missing Not At Random, white pixels are two times more likely to be missing. 

### 3.1. Compute statistics on the missing data given by the authors

In [4]:
# mask given as (S, V, T)
def compute_mask_stastics(mask): 
  s, v, t = mask.shape
  miss_per_img = np.sum(mask, axis=2).reshape(-1) / t
  img_avg, img_std = np.mean(miss_per_img), np.std(miss_per_img)

  miss_per_sample = np.sum(mask, axis=(2,1)) / (v*t)
  sample_avg, sample_std = np.mean(miss_per_sample), np.std(miss_per_sample)

  tot_miss_ratio = np.sum(mask) / (s*v*t)

  mask = mask.transpose(0,2,1)
  miss_per_series = np.sum(mask, axis=2)
  series_avg = np.mean(miss_per_series, axis=1)
  series_std = np.std(miss_per_series, axis=1)

  return ((img_avg, img_std), (sample_avg, sample_std), tot_miss_ratio, (np.mean(series_avg), np.mean(series_std)))

In [4]:
import numpy as np

hmnist_miss = np.load('external/datasets/hmnist/x_miss_nan.npy')
hmnist_miss = hmnist_miss.reshape(784, 70_000, 10).transpose(1,2,0)
mask = np.isnan(hmnist_miss)
hmnist_miss[mask] = 0.0

### 3.2. Authors' HMNIST MNAR

In [5]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'
validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

hmnist_miss_val = np.load("./external/datasets/hmnist/x_miss_nan.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]
hmnist_full_val_gt = np.load("./external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_miss_val)
# ts.normalize(normalizer="z_score")
ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts.data, 
                        "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                        model_folder,
                        ground_truth=ground_truth,
                        return_no_gt_imputation=True,
                        verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(hmnist_miss_val.T.reshape(-1, 10, 784))))
  
  create_imputation_plot((28,28,1),
                       10, 
                       ts.data.T.reshape(-1, seq_length, nbr_features),
                       ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                       ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                       ground_truth,
                       sample_idx)  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

Checkpoint successfully restored.


Imputing progress: 100%|██████████| 2/2 [00:02<00:00,  1.06s/it]


Model evaluation...
{'nll': 0.3571712882674497, 'mse': 0.11048427755163014}
((0.44792572766811645, 0.022531871365504545), (0.4479257276681164, 0.015794818474347896), 0.44792572766811645, (4.479257276681165, 1.7810154315420224))


### 3.3. Custom Contaminations

To contaminate effectively I need more data, seq_length of 10 is too small, already with the offset the start of the sequence is not contaminated. Solution, contaminate a larger set of data, and then select to visualize the imputation, in this way however, the nll and mse are not probably representative of the contamination. I think I could then save the contamination of the specific sample, and evaluate it individually.

Scattered

In [9]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'

validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

rate_dataset = 1.0
rate_series = 0.4
offset=0.1

hmnist_full_val_gt = np.load("./external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_full_val_gt)
# ts.normalize(normalizer="z_score")

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.scattered(ts.data,rate_dataset=rate_dataset,
                                  rate_series=rate_series, 
                                  offset=offset)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: SCATTER
	percentage of contaminated series: 100.0%
	rate of missing data per series: 40.0%
	security offset: [0-122]
	index impacted : 122 -> 610
Checkpoint successfully restored.


Imputing progress: 100%|██████████| 1/1 [00:00<00:00,  5.95it/s]


Model evaluation...
{'nll': 0.3758949775293875, 'mse': 0.11196729045447397}
((0.8110969387755101, 0.00175353661796781), (0.8110969387755103, 0.0), 0.8110969387755103, (8.110969387755102, 3.8447794022705795))


MCAR

In [7]:
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae
import numpy as np

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'

validation_idx = 600_000
sample_idx = 61
nr_samples_test = 122

# contamination parameters
rate_dataset = 1.0
rate_series = 0.8
block_size = 3

hmnist_full_val_gt = np.load("./external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_full_val_gt)
# ts.normalize(normalizer="z_score")

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data,
                             rate_dataset=rate_dataset, 
                             rate_series=rate_series, block_size=block_size)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: MCAR
	selected series: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 

Imputing progress: 100%|██████████| 1/1 [00:00<00:00,  8.43it/s]

Model evaluation...


{'nll': 0.6102993313294416, 'mse': 0.13838903170522707}
((0.8931122448979592, 0.007981881554674487), (0.8931122448979592, 0.0), 0.8931122448979592, (8.931122448979592, 1.9067149774977934))
